Evaluation & iteration notebook with Plotly visualizations comparing V1 vs V2 AI function accuracy
*Co-authored with CoCo*

# Notebook 3: Evaluation & Iteration

## Why Evaluation Matters

An AI function without evaluation is just a guess. Evaluation gives you:
- **Baseline metrics** — how well does the current version actually perform?
- **Field-level diagnostics** — which outputs are accurate and which need work?
- **Iteration evidence** — quantified proof that V2 is better than V1

## Probabilistic Nature of LLMs

LLMs are inherently non-deterministic. Even with temperature=0 and structured output constraints:
- Accuracy scores will fluctuate +/- 5-10% between runs on small datasets
- A "100% accuracy" on 12 test rows does NOT guarantee production performance
- Focus on **directional improvement** (V1 to V2), not absolute scores
- For production confidence, use 50-100+ labeled examples

---

In [ ]:
# Prerequisite check: Ensure Notebook 2 (Function Creation) has been run
from snowflake.snowpark.context import get_active_session
session = get_active_session()
try:
    state = session.sql("SELECT NOTEBOOK_ID FROM AI_STUDIO_DEMO.PUBLIC.DEMO_STATE WHERE NOTEBOOK_ID = 'function_creation'").collect()
    if not state:
        raise Exception('Not found')
    print('Prerequisite check passed: Notebook 2 (Function Creation) has been completed.')
except:
    print('WARNING: Please run Notebook 2 (AI Function Creation) first!')
    print('   That notebook creates the CLASSIFY_SUPPORT_TICKET and V2 functions required here.')

In [ ]:
%%sql -r dataframe_1
USE DATABASE AI_STUDIO_DEMO;
USE SCHEMA PUBLIC;

## Step 1: Create Labeled Test Data

Evaluation requires a table with:
- **Input column(s):** the ticket text fed to the function
- **Label column:** the expected/ground-truth output (human-verified classifications)

For multi-output functions like ours (RETURNS VARIANT), the label column is a VARIANT containing the expected JSON.

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE EVALUATION_TEST_DATA (
    TEST_ID INT,
    TICKET_TEXT VARCHAR,
    EXPECTED_OUTPUT VARIANT
);

In [ ]:
%%sql -r dataframe_3
-- 12 human-labeled ground truth examples
INSERT INTO EVALUATION_TEST_DATA
SELECT 1, 'Subject: URGENT - XT-400 bond failure on automotive trim assembly\n\nOur Tier 1 line in Monterrey has been down since 6am. The XT-400 tape is delaminating. This is blocking 2,200 units/day.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Product Defect / Failure","priority":"P1-Critical","customer_segment":"OEM","sla_flag":true}')
UNION ALL SELECT 2, 'Subject: CRITICAL - Respirator filter failure during chemical spill response\n\nDuring a chlorine gas response, workers reported breakthrough odor through cartridges. OSHA notified. Lives at stake.',
  PARSE_JSON('{"division":"Safety & Industrial","issue_type":"Product Defect / Failure","priority":"P1-Critical","customer_segment":"End User","sla_flag":true}')
UNION ALL SELECT 3, 'Subject: Production halt - thermal interface material failure\n\nAustin fab stopped production. TC-5026 thermal compound showing 3x spec resistance. $2.1M in WIP scrapped. $45M contract at risk.',
  PARSE_JSON('{"division":"Electronics & Energy","issue_type":"Product Defect / Failure","priority":"P1-Critical","customer_segment":"OEM","sla_flag":true}')
UNION ALL SELECT 4, 'Subject: MIL-SPEC compliance issue on reflective tape\n\nRetro-reflective sheeting does not meet MIL-PRF-680. DoD deadline in 3 weeks. Contract $1.2M, liquidated damages in 21 days.',
  PARSE_JSON('{"division":"Safety & Industrial","issue_type":"Regulatory / Compliance","priority":"P2-High","customer_segment":"Government/Defense","sla_flag":true}')
UNION ALL SELECT 5, 'Subject: Surgical tape adhesion failure across 3 facilities\n\n12 incidents of SecureSkin Plus tape lifting. Chief Medical Officer involved. Pulling lot from 47 facilities. FDA MedWatch filing.',
  PARSE_JSON('{"division":"Healthcare","issue_type":"Product Defect / Failure","priority":"P2-High","customer_segment":"End User","sla_flag":true}')
UNION ALL SELECT 6, 'Subject: Technical data sheet request - structural adhesives\n\nEvaluating adhesives for carbon fiber bicycle frame. Need TDS for Scotch-Weld AF163-2, DP460NS. Design phase, prototyping Q3.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Technical Specification Inquiry","priority":"P3-Standard","customer_segment":"OEM","sla_flag":false}')
UNION ALL SELECT 7, 'Subject: Requesting samples of cold-shrink splice kits\n\nPlanning to replace 45 splice joints on 15kV system. Need 3 samples of QS-II series. Project starts June 15th.',
  PARSE_JSON('{"division":"Electronics & Energy","issue_type":"Sample Request","priority":"P3-Standard","customer_segment":"End User","sla_flag":false}')
UNION ALL SELECT 8, 'Subject: Standard RMA for defective transfer tape\n\n4 of 20 rolls have contamination. Requesting RMA. Not urgent - sufficient inventory.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Warranty / RMA","priority":"P3-Standard","customer_segment":"Converter","sla_flag":false}')
UNION ALL SELECT 9, 'Subject: 2026 product catalog and pricing request\n\nNeed updated catalog and distributor price list. No rush - 2-3 weeks is fine.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Technical Specification Inquiry","priority":"P4-Low","customer_segment":"Distributor","sla_flag":false}')
UNION ALL SELECT 10, 'Subject: Future project scoping - adhesive for cobot joints\n\nEarly R&D, 18-month timeline. Bonding aluminum to PEEK. Just starting a conversation.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Application Engineering Support","priority":"P4-Low","customer_segment":"OEM","sla_flag":false}')
UNION ALL SELECT 11, 'Subject: Pricing dispute on last invoice\n\nInvoice shows $347/case for Command strips but contract says $312/case. Credit $2,940 difference.',
  PARSE_JSON('{"division":"Consumer","issue_type":"Pricing / Contract Dispute","priority":"P3-Standard","customer_segment":"End User","sla_flag":false}')
UNION ALL SELECT 12, 'Subject: REACH compliance declaration needed for 5 materials\n\nEU audit March 28. Missing REACH declarations for TC-5026, Novec 7100, VHB 5952. Need within 72 hours. $200M shipments at risk.',
  PARSE_JSON('{"division":"Electronics & Energy","issue_type":"Regulatory / Compliance","priority":"P2-High","customer_segment":"OEM","sla_flag":true}');

## Step 2: Evaluate V1 (Baseline)

We run `CLASSIFY_SUPPORT_TICKET` (V1) on every row in the test data and compare its output to the expected labels. Each field is scored independently using **exact string matching** — the function output for a field must match the ground truth character-for-character.

This per-field approach (rather than all-or-nothing) pinpoints exactly _which_ fields the model struggles with, guiding targeted prompt improvements.

In [ ]:
%%sql -r dataframe_4
-- V1 field-level accuracy
SELECT
  'V1 (baseline)' AS version,
  COUNT(*) AS total_tests,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:priority::VARCHAR = c:priority::VARCHAR, 1.0, 0.0)), 3) AS priority_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:division::VARCHAR = c:division::VARCHAR, 1.0, 0.0)), 3) AS division_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:issue_type::VARCHAR = c:issue_type::VARCHAR, 1.0, 0.0)), 3) AS issue_type_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:customer_segment::VARCHAR = c:customer_segment::VARCHAR, 1.0, 0.0)), 3) AS segment_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:sla_flag::BOOLEAN = c:sla_flag::BOOLEAN, 1.0, 0.0)), 3) AS sla_flag_acc
FROM EVALUATION_TEST_DATA t,
LATERAL (SELECT CLASSIFY_SUPPORT_TICKET(t.TICKET_TEXT) AS c);

### V1 Findings

Look at the results above — you'll see a pattern:
- **division & issue_type: ~90%+** — the model handles these well
- **sla_flag: ~90%** — binary yes/no is fairly reliable  
- **segment: ~80%** — decent but room to improve
- **priority: very low (~8%)** — critical failure! The model paraphrases instead of selecting from the enum

**Root cause:** Without strict output constraints, LLMs tend to use their own phrasing rather than matching expected enum values exactly. The priority field has several valid-sounding but non-matching alternatives (e.g., "High" vs "P2-High").

---

## Step 3: Evaluate V2 (Improved)

In V2, we added `response_format` with JSON schema that includes strict `enum` constraints. This forces the model to output only valid values for each classification field.

In [ ]:
%%sql -r dataframe_5
-- V2 field-level accuracy
SELECT
  'V2 (improved)' AS version,
  COUNT(*) AS total_tests,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:priority::VARCHAR = c:priority::VARCHAR, 1.0, 0.0)), 3) AS priority_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:division::VARCHAR = c:division::VARCHAR, 1.0, 0.0)), 3) AS division_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:issue_type::VARCHAR = c:issue_type::VARCHAR, 1.0, 0.0)), 3) AS issue_type_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:customer_segment::VARCHAR = c:customer_segment::VARCHAR, 1.0, 0.0)), 3) AS segment_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:sla_flag::BOOLEAN = c:sla_flag::BOOLEAN, 1.0, 0.0)), 3) AS sla_flag_acc
FROM EVALUATION_TEST_DATA t,
LATERAL (SELECT CLASSIFY_SUPPORT_TICKET_V2(t.TICKET_TEXT) AS c);

### V2 Results

Compare V2's numbers to V1 above — the key improvement is in **priority accuracy**, which jumps dramatically thanks to enum constraints in the `response_format` schema.

**Key takeaway:** `response_format` enum constraints provide **token-level enforcement** that prompt instructions alone cannot guarantee. The model _must_ choose from the valid values, eliminating paraphrasing errors.

### Visual Comparison: V1 vs V2

The bar chart below shows field-level accuracy side-by-side, while the radar chart reveals the overall "shape" of each version's performance profile. A larger, more uniform radar polygon indicates a more balanced classifier.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fields = ['Priority', 'Division', 'Issue Type', 'Segment', 'SLA Flag']
cols = ['PRIORITY_ACC', 'DIVISION_ACC', 'ISSUE_TYPE_ACC', 'SEGMENT_ACC', 'SLA_FLAG_ACC']
v1_scores = [float(dataframe_4.iloc[0][c]) for c in cols]
v2_scores = [float(dataframe_5.iloc[0][c]) for c in cols]

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'xy'}, {'type': 'polar'}]],
    subplot_titles=('Field-Level Accuracy', 'Performance Profile'),
    column_widths=[0.55, 0.45]
)

# Grouped bar chart
fig.add_trace(
    go.Bar(name='V1 (Baseline)', x=fields, y=v1_scores,
           marker_color='#FF6B6B', text=[f'{s:.0%}' for s in v1_scores],
           textposition='outside'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(name='V2 (Improved)', x=fields, y=v2_scores,
           marker_color='#4ECDC4', text=[f'{s:.0%}' for s in v2_scores],
           textposition='outside'),
    row=1, col=1
)

# Radar chart
radar_fields = fields + [fields[0]]
v1_radar = v1_scores + [v1_scores[0]]
v2_radar = v2_scores + [v2_scores[0]]

fig.add_trace(
    go.Scatterpolar(r=v1_radar, theta=radar_fields, fill='toself',
                    name='V1 (Baseline)', line_color='#FF6B6B',
                    fillcolor='rgba(255,107,107,0.2)'),
    row=1, col=2
)
fig.add_trace(
    go.Scatterpolar(r=v2_radar, theta=radar_fields, fill='toself',
                    name='V2 (Improved)', line_color='#4ECDC4',
                    fillcolor='rgba(78,205,196,0.2)'),
    row=1, col=2
)

fig.update_layout(
    title_text='V1 vs V2 Evaluation Comparison',
    height=450, width=1000,
    template='plotly_white',
    showlegend=True,
    legend=dict(x=0.35, y=-0.15, orientation='h')
)
fig.update_yaxes(range=[0, 1.15], tickformat='.0%', row=1, col=1)
fig.update_polars(radialaxis=dict(range=[0, 1], tickformat='.0%'))
fig.update_layout(barmode='group')
fig.show()

---

## Step 4: Custom Evaluation Metric — Weighted Scoring

Not all fields are equally important to the business. A misclassified **priority** (which drives SLA timers and escalation) is far more costly than a misclassified segment. We define a **weighted accuracy metric** that reflects business impact:

| Field | Weight | Rationale |
|-------|--------|-----------|
| Priority | 0.4 | Drives SLA escalation and response times |
| Division | 0.3 | Routes to correct engineering team |
| Issue Type | 0.2 | Determines playbook / resolution path |
| SLA Flag | 0.1 | Binary derivative of priority + segment |

The function below returns a score between 0.0 and 1.0 plus a per-field PASS/FAIL breakdown.

In [ ]:
%%sql -r dataframe_6
-- Custom metric: weighted field accuracy
-- Priority 40%, Division 30%, Issue Type 20%, SLA Flag 10%
CREATE OR REPLACE FUNCTION WEIGHTED_FIELD_ACCURACY(
    EXPECTED VARCHAR, 
    PREDICTED VARCHAR
)
RETURNS OBJECT
LANGUAGE SQL
AS
$$
SELECT OBJECT_CONSTRUCT(
  'score',
    (CASE WHEN PARSE_JSON(EXPECTED):priority::VARCHAR = PARSE_JSON(PREDICTED):priority::VARCHAR THEN 0.4 ELSE 0.0 END) +
    (CASE WHEN PARSE_JSON(EXPECTED):division::VARCHAR = PARSE_JSON(PREDICTED):division::VARCHAR THEN 0.3 ELSE 0.0 END) +
    (CASE WHEN PARSE_JSON(EXPECTED):issue_type::VARCHAR = PARSE_JSON(PREDICTED):issue_type::VARCHAR THEN 0.2 ELSE 0.0 END) +
    (CASE WHEN PARSE_JSON(EXPECTED):sla_flag::BOOLEAN = PARSE_JSON(PREDICTED):sla_flag::BOOLEAN THEN 0.1 ELSE 0.0 END),
  'feedback',
    CONCAT(
      'Priority: ', IFF(PARSE_JSON(EXPECTED):priority::VARCHAR = PARSE_JSON(PREDICTED):priority::VARCHAR, 'PASS', 'FAIL'),
      ' | Division: ', IFF(PARSE_JSON(EXPECTED):division::VARCHAR = PARSE_JSON(PREDICTED):division::VARCHAR, 'PASS', 'FAIL'),
      ' | Issue Type: ', IFF(PARSE_JSON(EXPECTED):issue_type::VARCHAR = PARSE_JSON(PREDICTED):issue_type::VARCHAR, 'PASS', 'FAIL'),
      ' | SLA Flag: ', IFF(PARSE_JSON(EXPECTED):sla_flag::BOOLEAN = PARSE_JSON(PREDICTED):sla_flag::BOOLEAN, 'PASS', 'FAIL')
    )
)
$$;

In [ ]:
%%sql -r dataframe_7
-- Test the custom metric
SELECT 
  r['score']::FLOAT AS score,
  r['feedback']::VARCHAR AS feedback
FROM (
  SELECT WEIGHTED_FIELD_ACCURACY(
    '{"division":"Safety & Industrial","priority":"P1-Critical","issue_type":"Product Defect / Failure","sla_flag":true}',
    '{"division":"Safety & Industrial","priority":"P2-High","issue_type":"Product Defect / Failure","sla_flag":false}'
  ) AS r
);

### Understanding the Weighted Score

The test above shows how the metric works on a single example: Division and Issue Type both matched (0.3 + 0.2 = 0.5), but Priority and SLA Flag did not (missing 0.4 + 0.1 = 0.5). The resulting weighted score is **0.5 / 1.0**.

Below, we compute the weighted score across all test cases for both V1 and V2 to see the business-impact-adjusted improvement.

In [ ]:
import plotly.graph_objects as go

# Compute weighted scores using the same weights as WEIGHTED_FIELD_ACCURACY
weight_labels = ['Priority<br>(×0.4)', 'Division<br>(×0.3)', 'Issue Type<br>(×0.2)', 'SLA Flag<br>(×0.1)']
acc_cols = ['PRIORITY_ACC', 'DIVISION_ACC', 'ISSUE_TYPE_ACC', 'SLA_FLAG_ACC']
weight_vals = [0.4, 0.3, 0.2, 0.1]

v1_contribs = [float(dataframe_4.iloc[0][c]) * w for c, w in zip(acc_cols, weight_vals)]
v2_contribs = [float(dataframe_5.iloc[0][c]) * w for c, w in zip(acc_cols, weight_vals)]
v1_total = sum(v1_contribs)
v2_total = sum(v2_contribs)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='V1 (Baseline)', x=weight_labels, y=v1_contribs,
    marker_color='#FF6B6B', text=[f'{c:.3f}' for c in v1_contribs],
    textposition='outside'
))
fig.add_trace(go.Bar(
    name='V2 (Improved)', x=weight_labels, y=v2_contribs,
    marker_color='#4ECDC4', text=[f'{c:.3f}' for c in v2_contribs],
    textposition='outside'
))

fig.add_annotation(
    x=0.5, y=1.08, xref='paper', yref='paper',
    text=f'<b>Weighted Total Score — V1: {v1_total:.3f} | V2: {v2_total:.3f} | Improvement: +{v2_total - v1_total:.3f}</b>',
    showarrow=False, font=dict(size=13)
)

fig.update_layout(
    title='Weighted Score Breakdown: Business-Impact View',
    yaxis_title='Weighted Contribution to Score',
    yaxis=dict(range=[0, 0.48]),
    barmode='group',
    template='plotly_white',
    height=420,
    legend=dict(x=0.01, y=0.95)
)
fig.show()

The chart above shows how each field contributes to the business-weighted total. Priority — the highest-weighted field at 0.4 — is where V2 shows its biggest improvement. The `response_format` enum fix directly addressed the most impactful field.

---

## Step 5: Formal Evaluation via `EVALUATE_AI_FUNCTION()`

For production workflows, Snowflake provides `SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION()` — a stored procedure that:
1. Runs your AI function against every row in the test table
2. Evaluates predictions using your chosen method (LLM judge, custom metric, etc.)
3. Returns an aggregate score and stores per-row details in a **Snowflake Experiment**

This creates an auditable, version-tracked record of each evaluation run — essential for governance and iteration tracking.

In [ ]:
%%sql -r dataframe_8
-- Evaluate V2 with LLM Judge (best for multi-field VARIANT output)
CALL SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_SUPPORT_TICKET_V2',
    'AI_STUDIO_DEMO.PUBLIC.EVALUATION_TEST_DATA',
    ARRAY_CONSTRUCT('TICKET_TEXT'),
    'EXPECTED_OUTPUT',
    'llm_judge',
    'claude-sonnet-4-6',
    NULL,
    NULL,
    PARSE_JSON('{"task_description": "Classify enterprise B2B support tickets. Output must correctly identify division, issue_type, priority, customer_segment, and sla_flag."}'),
    500,
    NULL,
    NULL
);

---
## Summary: Iteration Through Evaluation

This notebook demonstrated the core evaluation loop for AI functions:

1. **Baseline measurement** — establish where V1 stands across all output fields
2. **Targeted fix** — use `response_format` enums to enforce valid values for the weakest field
3. **Re-evaluate** — confirm V2 improves the target field without regressing others
4. **Weighted scoring** — apply business-priority weights to get a single meaningful metric
5. **Formal tracking** — use `EVALUATE_AI_FUNCTION()` for auditable experiment records

The visual comparisons above make it clear: V2 is strictly better than V1, with the largest gains in the highest-weighted field (priority). This pattern — measure, fix, re-measure, visualize — scales to any number of iterations.

Proceed to **Notebook 4: Model Comparison & RBAC** to compare multiple LLMs and secure the function.

In [ ]:
%%sql -r dataframe_9
-- Mark this notebook as completed
MERGE INTO DEMO_STATE AS t
USING (SELECT 'evaluation' AS NOTEBOOK_ID, CURRENT_TIMESTAMP() AS COMPLETED_AT) AS s
ON t.NOTEBOOK_ID = s.NOTEBOOK_ID
WHEN MATCHED THEN UPDATE SET COMPLETED_AT = s.COMPLETED_AT
WHEN NOT MATCHED THEN INSERT (NOTEBOOK_ID, COMPLETED_AT) VALUES (s.NOTEBOOK_ID, s.COMPLETED_AT);